In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import re


In [ ]:
!pip uninstall -y keras

In [ ]:
!pip uninstall -y keras

In [ ]:
df = pd.read_csv('pythonversion/sentiment_classifier/datasets/IMDB Dataset.csv')
print(df.head())

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation/numbers
    return text

df['cleaned_review'] = df['review'].apply(clean_text)


# Remove empty reviews
df = df[df['cleaned_review'].str.strip() != '']

# Verify
print(f"Empty reviews removed. New dataset size: {len(df)}")

# -----------------------------------------------------
# 2. Convert Sentiment to Numerical Values (No Changes)
# -----------------------------------------------------
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# -----------------------------------------------------
# 3. Split Data (No Changes)
# -----------------------------------------------------
X = df['cleaned_review']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -----------------------------------------------------
# 4. Tokenization and Padding (Fixed)
# -----------------------------------------------------
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Define vocab_size and max_length
vocab_size = tokenizer.num_words + 1  # +1 for OOV token
train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

# Set max_length dynamically or to a fixed value
max_length = 200  # Or use: max(len(seq) for seq in train_sequences)
X_train_padded = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
X_test_padded = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

print(f"Vocabulary Size: {vocab_size}")
print(f"Max Sequence Length: {max_length}")


In [ ]:
model = Sequential([
    Embedding(vocab_size, 128),  # Reduced embedding dim
    Dropout(0.3),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),        # Fewer units
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer=Adam(clipvalue=1.0), metrics=['accuracy'])
model.summary()

In [ ]:
print("Training data shape:", X_train_padded.shape)
print("Sample padded sequence:", X_train_padded[0])

In [ ]:
history = model.fit(
    X_train_padded,
    y_train,
    epochs=10,
    batch_size=128,  
    validation_split=0.2
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Over Epochs')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Test set evaluation
test_loss, test_acc = model.evaluate(X_test_padded, y_test)
print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Confusion Matrix
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
sample_text = ["This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout.", 
               "Terrible waste of time. Poor acting and a nonsensical storyline."]

# Preprocess
sample_sequences = tokenizer.texts_to_sequences(sample_text)
sample_padded = pad_sequences(sample_sequences, maxlen=max_length, padding='post', truncating='post')

# Predict
predictions = (model.predict(sample_padded) > 0.5).astype("int32")
sentiments = ['positive' if pred == 1 else 'negative' for pred in predictions]

for text, sentiment in zip(sample_text, sentiments):
    print(f"Text: {text[:60]}...\nPredicted Sentiment: {sentiment}\n")